# Customers table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).

Decisions and findings for this table are written up in `cleaning_notes/README_customers.md`.

### 1. Imports and settings

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw customers file

In [ ]:
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')

### 3. First inspection: structure, nulls, preview

In [ ]:
customers.info()
print(customers.isna().sum())      # expect all zeros
print(customers.head())

### 4. Key analysis: customer_id is per order, customer_unique_id is per person

In [ ]:
print(customers['customer_id'].is_unique)                  # expect True
print(customers['customer_id'].nunique())                  # 99,441
print(customers['customer_unique_id'].nunique())           # 96,096

orders_per_person = customers['customer_unique_id'].value_counts()
print(orders_per_person.head())                            # top buyer has 17 orders
print((orders_per_person > 1).sum())                       # 2,997 repeat customers
print((orders_per_person > 1).mean() * 100)                # about 3.12 (%)

### 5. Join integrity with orders (both directions)

In [ ]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['customer_id'])
print(orders['customer_id'].isin(customers['customer_id']).value_counts())   # True 99,441
print(customers['customer_id'].isin(orders['customer_id']).all())            # True

### 6. Zip code prefixes: restore leading zeros (stored as text, 5 characters)

In [ ]:
customers['customer_zip_code_prefix'] = (
    customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
)
print(customers['customer_zip_code_prefix'].str.len().value_counts())        # only 5

### 7. State and city checks

In [ ]:
print(customers['customer_state'].nunique())               # 27
print((customers['customer_city'] != customers['customer_city'].str.strip()).sum())   # 0 stray spaces
print(customers.loc[customers['customer_city'].str.contains(r"[^a-z \-']", regex=True),
                    'customer_city'].unique())             # only 'quilometro 14 do mutum'

# City casing for display: 'sao paulo' -> 'Sao Paulo'
customers['customer_city'] = customers['customer_city'].str.title()
print(customers['customer_city'].nunique())                # still 4,119

### 8. Save the cleaned table and verify the file on disk

In [ ]:
customers.to_csv(CLEAN / 'customers_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'customers_clean.csv', dtype={'customer_zip_code_prefix': str})
print(check.shape)                                         # expect (99441, 5)
print(check['customer_zip_code_prefix'].str.len().value_counts())   # only 5
print(check['customer_city'].nunique())                    # 4,119